# Preset 參考圖（Colab / Z-Image-Turbo，免登入、不截斷 prompt）

用 Colab 免費 T4 替 Prompt Builder 的 310 個 preset 逐一生圖，作為第二家參考圖（來源標示 `colab`）。

**模型**：`Tongyi-MAI/Z-Image-Turbo`（阿里通義，6B，Apache-2.0，2026-10-03 以 HF API 查證 gated=False：免登入、不需 token、不分享任何資料）。中英文字樣支援較好；文字編碼器 prompt 上限 512 tokens（SDXL 只有 77），**本筆記本送進模型的是原 prompt，不清理、不截斷**。

**為什麼分兩階段**：免費 T4 只有 12.7 GB 系統 RAM、15 GB 顯存，文字編碼器（約 8 GB）加 transformer（約 12 GB）放不下。所以先只載編碼器把這一批 prompt 編碼完並釋放，再載 transformer 與 VAE 生圖。

**用法**：執行階段選 **T4 GPU** → 依序執行 → 先看「煙霧測試」再跑「批次」。每次批次最多 5 張；煙霧測試與重複批次合計硬上限 6 次，失敗也計次。
**可續跑**：進度記在 `manifest.json`，斷線後重跑「批次」即可；已成功者不重做。
**輸入**：已內嵌於第 2 格（presets_prompts.json 與 AGY 已有圖清單），**不需上傳檔案、不掛載 Drive**。
**輸出**：`prompt_builder_colab/preset-originals/colab/*.png`（原始尺寸、不縮放、無浮水印）與 `preset-previews/colab/manifest.json`（結構同 Codex／AGY）。一律存 `/content/prompt_builder_colab`（斷線會消失），批次後請用最後一格 zip 下載。

**限制（未在你帳號實測）**：
- 免費 T4 沒有原生 bf16，會較慢（第三方文章估約 1 分鐘／張，未驗證）。
- 單次執行階段上限約 3 小時 20 分（Colab 資源面板顯示），GPU 取得不保證。
- 帳號目前無運算單位、無付費方案；本筆記本只使用免費資源，不購買、不升級。
- 開源模型，風格與品質與 Gemini／Codex 不同。

In [ ]:
# 1) 環境檢查與安裝
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout or "找不到 GPU：請把執行階段類型改成 T4 GPU")
pip = [sys.executable, "-m", "pip", "install", "-q", "-U"]
subprocess.check_call(pip + ["transformers", "accelerate", "safetensors", "sentencepiece", "diffusers"])
r = subprocess.run([sys.executable, "-c", "from diffusers import ZImagePipeline"], capture_output=True, text=True)
if r.returncode != 0:   # 發行版尚無 ZImagePipeline 時，改裝 GitHub main（模型卡指示的做法）
    print("發行版 diffusers 無 ZImagePipeline，改裝 GitHub main …")
    subprocess.check_call(pip + ["git+https://github.com/huggingface/diffusers"])
print("diffusers 就緒")

In [ ]:
# 2) 設定與輸入（輸入檔已內嵌於本筆記本，不需上傳、不掛載 Drive）
import os, json, hashlib, re, time, gzip, base64

ROOT = "/content/prompt_builder_colab"   # 本機暫存；斷線會消失，批次後請用最後一格打包下載
ORIG = f"{ROOT}/preset-originals/colab"
PREV = f"{ROOT}/preset-previews/colab"
MANIFEST = f"{PREV}/manifest.json"
os.makedirs(ORIG, exist_ok=True); os.makedirs(PREV, exist_ok=True)

MAX_PIXELS = 1024 * 1024   # 目標像素數
MAX_NEW_PER_RUN = 5        # 保留煙霧測試1張；全輪總量仍由下方硬上限守門
MAX_IMAGE_CALLS = 6       # 本輪已用6次模型呼叫；未重新授權前不得提高
CALL_BUDGET = f"{ROOT}/call-budget.json"
MAX_CONSEC_ERRORS = 1      # 連續失敗幾次就停

_B64 = (
    "..."
)
_RAW = gzip.decompress(base64.b64decode(_B64))
assert hashlib.sha256(_RAW).hexdigest() == "ccd69c22c611a73a9de2354984a590be88ef25cf39f0e89c50706850df168a34", "內嵌 presets_prompts.json 校驗失敗"
PRESETS = json.loads(_RAW.decode("utf-8"))
# AGY 已有圖（manifest 為 ok）者跳過，節省運算
SKIP_KEYS = set(["台灣風格 (Taiwan)/tw_hipster_cafe", "台灣風格 (Taiwan)/tw_modern_poetry", "台灣風格 (Taiwan)/tw_new_wave", "台灣風格 (Taiwan)/tw_night_market", "風格示範 攝影題材 (Photography Genres)/style_example_1_e_commerce_photography", "風格示範 攝影題材 (Photography Genres)/style_example_1_underwater_photography", "風格示範 攝影題材 (Photography Genres)/style_example_1_wildlife_photography", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_cinestill_50d", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_fujifilm_pro_400h", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_fujifilm_velvia_50", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_kodak_ektar_100", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_kodak_gold_200", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_kodak_tri_x_400", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_long_exposure", "風格示範 相機與鏡頭 (Camera & Gear)/style_example_2_telephoto_lens", "風格示範 藝術風格 (Art Styles)/style_example_0_cel_shading", "風格示範 藝術風格 (Art Styles)/style_example_0_collage_art", "風格示範 藝術風格 (Art Styles)/style_example_0_impressionism", "風格示範 藝術風格 (Art Styles)/style_example_0_ink_wash_painting", "風格示範 藝術風格 (Art Styles)/style_example_0_kawaii_style", "風格示範 藝術風格 (Art Styles)/style_example_0_linocut", "風格示範 藝術風格 (Art Styles)/style_example_0_makoto_shinkai", "風格示範 藝術風格 (Art Styles)/style_example_0_oil_painting", "風格示範 藝術風格 (Art Styles)/style_example_0_pastel_drawing", "風格示範 藝術風格 (Art Styles)/style_example_0_pixar_animation_style", "風格示範 藝術風格 (Art Styles)/style_example_0_pixel_art", "風格示範 藝術風格 (Art Styles)/style_example_0_screen_print", "風格示範 藝術風格 (Art Styles)/style_example_0_ukiyo_e_style", "🎓 醫學教材與圖解 (Medical Education & Diagrams)/clinic_website_hero", "🎓 醫學教材與圖解 (Medical Education & Diagrams)/kids_brave_sticker", "🎓 醫學教材與圖解 (Medical Education & Diagrams)/med_anatomy_cutaway", "🎓 醫學教材與圖解 (Medical Education & Diagrams)/med_flowchart_infographic", "🎨 插畫與動漫 (Illustration & Anime)/ai_avatar", "🎨 插畫與動漫 (Illustration & Anime)/ai_character_sheet", "🎨 插畫與動漫 (Illustration & Anime)/ai_seamless_pattern", "🎨 插畫與動漫 (Illustration & Anime)/anime_city_pop", "🎨 插畫與動漫 (Illustration & Anime)/anime_jojo", "🎨 插畫與動漫 (Illustration & Anime)/anime_kyoani", "🎨 插畫與動漫 (Illustration & Anime)/game_char", "🎨 插畫與動漫 (Illustration & Anime)/game_iso", "🎨 插畫與動漫 (Illustration & Anime)/gf_robot_cinema", "🎨 插畫與動漫 (Illustration & Anime)/gf_robot_cyber", "🎨 插畫與動漫 (Illustration & Anime)/gf_robot_ghibli", "🎨 插畫與動漫 (Illustration & Anime)/gf_robot_macro", "🎨 插畫與動漫 (Illustration & Anime)/lego_city", "🎨 插畫與動漫 (Illustration & Anime)/manga_horror", "🎨 插畫與動漫 (Illustration & Anime)/manga_retro_90s", "🎨 插畫與動漫 (Illustration & Anime)/manga_shojo", "🎨 插畫與動漫 (Illustration & Anime)/manga_shonen", "🎨 插畫與動漫 (Illustration & Anime)/manga_webtoon", "🎨 插畫與動漫 (Illustration & Anime)/travel_blueprint_museum", "🎨 插畫與動漫 (Illustration & Anime)/travel_botanical_ink", "🎨 插畫與動漫 (Illustration & Anime)/travel_cafe_moment", "🎨 插畫與動漫 (Illustration & Anime)/travel_food_diary", "🎨 插畫與動漫 (Illustration & Anime)/travel_kyoto_sketch", "🎨 插畫與動漫 (Illustration & Anime)/travel_landscape_charcoal", "🎨 插畫與動漫 (Illustration & Anime)/travel_map_doodle", "🎨 插畫與動漫 (Illustration & Anime)/travel_postcard_gouache", "🎨 插畫與動漫 (Illustration & Anime)/tw_majolica_tile", "🎨 插畫與動漫 (Illustration & Anime)/tw_temple_dragon", "🏥 臨床與診所素材 (Clinic Assets)/clinic_fb_cover", "🏥 臨床與診所素材 (Clinic Assets)/friendly_staff", "🏥 臨床與診所素材 (Clinic Assets)/medical_slide_bg", "📐 UI 與排版 (UI & Layout)/complex_rpg_hud", "📐 UI 與排版 (UI & Layout)/complex_timeline", "📐 UI 與排版 (UI & Layout)/design_swiss_system", "📐 UI 與排版 (UI & Layout)/layout_central_hero", "📐 UI 與排版 (UI & Layout)/layout_collage_art", "📐 UI 與排版 (UI & Layout)/layout_frame_in_frame", "📐 UI 與排版 (UI & Layout)/layout_infographic_step", "📐 UI 與排版 (UI & Layout)/layout_isometric_room", "📐 UI 與排版 (UI & Layout)/layout_mag_cover", "📐 UI 與排版 (UI & Layout)/layout_slide_bg", "📐 UI 與排版 (UI & Layout)/layout_split_screen", "📐 UI 與排版 (UI & Layout)/layout_typographic_poster", "📐 UI 與排版 (UI & Layout)/slide_roadmap", "📐 UI 與排版 (UI & Layout)/slide_team", "📐 UI 與排版 (UI & Layout)/slide_tech_dark", "📐 UI 與排版 (UI & Layout)/ui_app_fin", "📐 UI 與排版 (UI & Layout)/ui_dashboard_dark", "📐 UI 與排版 (UI & Layout)/ui_ecommerce", "📐 UI 與排版 (UI & Layout)/ui_landing_page", "📐 UI 與排版 (UI & Layout)/ui_mobile_game", "📐 UI 與排版 (UI & Layout)/ui_neobrutalism", "📐 UI 與排版 (UI & Layout)/ui_web_hero", "📐 經典切版佈局 (Classic Layouts)/complex_magazine_spread", "📐 經典切版佈局 (Classic Layouts)/layout_landing_page", "📐 經典切版佈局 (Classic Layouts)/layout_movie_poster", "📚 教育與解說 (Education & Info)/edu_aircraft_carrier", "📚 教育與解說 (Education & Info)/edu_flashcard", "📚 教育與解說 (Education & Info)/edu_mindmap", "📚 教育與解說 (Education & Info)/edu_poster", "📚 教育與解說 (Education & Info)/edu_quiz", "📚 教育與解說 (Education & Info)/edu_slide", "📚 教育與解說 (Education & Info)/edu_textbook", "📚 教育與解說 (Education & Info)/edu_worksheet", "📚 教育與解說 (Education & Info)/he_poster_dental", "📚 教育與解說 (Education & Info)/he_poster_diet", "📚 教育與解說 (Education & Info)/he_poster_mental", "📚 教育與解說 (Education & Info)/he_poster_senior", "📚 教育與解說 (Education & Info)/he_poster_vaccine", "📚 教育與解說 (Education & Info)/info_chart_universal", "📚 教育與解說 (Education & Info)/info_mindfulness", "📚 教育與解說 (Education & Info)/info_nutrition_chart", "📚 教育與解說 (Education & Info)/info_workout_routine", "📚 教育與解說 (Education & Info)/med_anatomy", "📚 教育與解說 (Education & Info)/med_clinical", "📚 教育與解說 (Education & Info)/med_consult", "📚 教育與解說 (Education & Info)/med_seminar", "📚 教育與解說 (Education & Info)/note_blackboard_math", "📚 教育與解說 (Education & Info)/note_bullet_journal", "📚 教育與解說 (Education & Info)/note_mindmap", "📚 教育與解說 (Education & Info)/note_recipe_steps", "📚 教育與解說 (Education & Info)/note_study_bio", "📚 教育與解說 (Education & Info)/sci_formula", "📚 教育與解說 (Education & Info)/sci_lab_diagram", "📚 教育與解說 (Education & Info)/sci_microscope", "📚 教育與解說 (Education & Info)/sci_poster", "📚 教育與解說 (Education & Info)/sci_safety", "📢 商業與社群 (Business & Social)/biz_card", "📢 商業與社群 (Business & Social)/ecommerce_banner", "📢 商業與社群 (Business & Social)/ecommerce_comparison", "📢 商業與社群 (Business & Social)/ecommerce_lifestyle", "📢 商業與社群 (Business & Social)/ecommerce_white_bg", "📢 商業與社群 (Business & Social)/event_badge", "📢 商業與社群 (Business & Social)/event_birthday", "📢 商業與社群 (Business & Social)/event_certificate", "📢 商業與社群 (Business & Social)/event_savedate", "📢 商業與社群 (Business & Social)/event_ticket", "📢 商業與社群 (Business & Social)/event_wedding", "📢 商業與社群 (Business & Social)/food_cafe_board", "📢 商業與社群 (Business & Social)/food_menu", "📢 商業與社群 (Business & Social)/food_photography", "📢 商業與社群 (Business & Social)/food_recipe", "📢 商業與社群 (Business & Social)/layout_biz_poster", "📢 商業與社群 (Business & Social)/social_carousel", "📢 商業與社群 (Business & Social)/social_fb_event", "📢 商業與社群 (Business & Social)/social_ig_story", "📢 商業與社群 (Business & Social)/social_lifestyle_vibe", "📢 商業與社群 (Business & Social)/social_linkedin_award", "📢 商業與社群 (Business & Social)/social_linkedin_cover", "📢 商業與社群 (Business & Social)/social_pinterest_diy", "📢 商業與社群 (Business & Social)/social_tiktok_cover", "📢 商業與社群 (Business & Social)/social_yt_thumb", "📦 樣機與展示 (Mockups & Showcase)/mockup_cosmetic", "📦 樣機與展示 (Mockups & Showcase)/mockup_poster", "📦 樣機與展示 (Mockups & Showcase)/mockup_smartphone", "📦 樣機與展示 (Mockups & Showcase)/mockup_tshirt", "📸 攝影、電影與寫真 (Photography & Portraits)/arch_future", "📸 攝影、電影與寫真 (Photography & Portraits)/arch_loft", "📸 攝影、電影與寫真 (Photography & Portraits)/arch_modern", "📸 攝影、電影與寫真 (Photography & Portraits)/arch_scandi", "📸 攝影、電影與寫真 (Photography & Portraits)/arch_zen", "📸 攝影、電影與寫真 (Photography & Portraits)/art_riso", "📸 攝影、電影與寫真 (Photography & Portraits)/movie_action", "📸 攝影、電影與寫真 (Photography & Portraits)/movie_horror", "📸 攝影、電影與寫真 (Photography & Portraits)/movie_scifi_epic", "📸 攝影、電影與寫真 (Photography & Portraits)/nature_macro", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_abstract", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_bw_portrait", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_cinestill", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_film", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_film_portra", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_instant", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_rembrandt", "📸 攝影、電影與寫真 (Photography & Portraits)/photo_street", "📸 攝影、電影與寫真 (Photography & Portraits)/port_cinematic", "📸 攝影、電影與寫真 (Photography & Portraits)/port_env", "📸 攝影、電影與寫真 (Photography & Portraits)/port_fashion", "📸 攝影、電影與寫真 (Photography & Portraits)/port_street", "📸 攝影、電影與寫真 (Photography & Portraits)/port_studio", "📸 攝影、電影與寫真 (Photography & Portraits)/prod_gobo", "📸 攝影、電影與寫真 (Photography & Portraits)/prod_perfume", "📸 攝影、電影與寫真 (Photography & Portraits)/prod_watch", "📸 攝影、電影與寫真 (Photography & Portraits)/tw_jiufen_lanterns", "📸 攝影、電影與寫真 (Photography & Portraits)/tw_tea_plantation", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_app_login", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_dashboard", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_diagonal_clash", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_linkedin", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_meme", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_podcast", "🔥 十大佈局實戰範本 (Top 10 Layouts)/layout_pyramid", "🔰 新手啟發範本 (Starter Inspiration)/crayon_park_play", "🔰 新手啟發範本 (Starter Inspiration)/starter_fisheye", "🔰 新手啟發範本 (Starter Inspiration)/starter_glitch", "🔰 新手啟發範本 (Starter Inspiration)/starter_isometric", "🔰 新手啟發範本 (Starter Inspiration)/starter_macro", "🔰 新手啟發範本 (Starter Inspiration)/starter_panoramic_symmetry", "🔰 新手啟發範本 (Starter Inspiration)/starter_papercraft", "🧸 兒科衛教 (Pediatric Education)/he_fb_baby_hero", "🧸 兒科衛教 (Pediatric Education)/he_fb_handwash", "🧸 兒科衛教 (Pediatric Education)/he_fb_watercolor", "🧸 兒科衛教 (Pediatric Education)/pedu_mechanism", "🧸 兒科衛教 (Pediatric Education)/pedu_parent", "🧸 兒科衛教 (Pediatric Education)/pedu_portrait", "🧸 兒科衛教 (Pediatric Education)/pedu_principles", "🧸 兒科衛教 (Pediatric Education)/pedu_scene", "🧸 兒科衛教 (Pediatric Education)/pedu_steps"])
print(len(PRESETS), "presets；跳過", len(SKIP_KEYS), "個 AGY 已有圖者")

In [ ]:
# 3) 輔助函式（純 Python，不依賴 GPU）
def parse_ratio(prompt):
    m = re.search(r"aspect ratio\s*(\d+)\s*:\s*(\d+)", prompt)
    return (int(m.group(1)), int(m.group(2))) if m else (1, 1)

def size_for(ratio, max_pixels=MAX_PIXELS, long_cap=1536, mult=16):
    rw, rh = ratio
    h = (max_pixels * rh / rw) ** 0.5
    w = h * rw / rh
    scale = min(1.0, long_cap / max(w, h))
    w, h = w * scale, h * scale
    return max(mult, int(round(w / mult)) * mult), max(mult, int(round(h / mult)) * mult)

def safe(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

def filename_for(p):
    return f"i{int(p['index']):03d}__{safe(p['key'])}.png"

def manifest_key(p):
    return f"{p['group']}/{p['key']}"

def load_manifest():
    if os.path.exists(MANIFEST):
        try:
            return json.load(open(MANIFEST, encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_manifest(m):
    tmp = MANIFEST + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=2)
    os.replace(tmp, MANIFEST)

def sha(prompt):
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()
# 煙霧測試與重複批次共用；失敗也消耗一次，重新執行設定格不清零。
def image_budget_state():
    if os.path.exists(CALL_BUDGET):
        with open(CALL_BUDGET, encoding="utf-8") as fh:
            state = json.load(fh)
        if not isinstance(state, dict) or type(state.get("used")) is not int or type(state.get("limit")) is not int:
            raise RuntimeError("生圖預算檔格式錯誤；先查原因，不重設")
        if state["used"] < 0 or state["limit"] < 1:
            raise RuntimeError("生圖預算檔數值錯誤")
        state["limit"] = min(state["limit"], MAX_IMAGE_CALLS)
        return state
    return {"used": 0, "limit": MAX_IMAGE_CALLS}

def remaining_image_calls():
    state = image_budget_state()
    return max(0, state["limit"] - state["used"])

def claim_image_call():
    state = image_budget_state()
    if state["used"] >= state["limit"]:
        raise RuntimeError(f"已達本輪生圖硬上限 {state['limit']} 次；停止，不重試")
    state["used"] += 1
    with open(CALL_BUDGET + ".tmp", "w", encoding="utf-8") as fh:
        json.dump(state, fh)
    os.replace(CALL_BUDGET + ".tmp", CALL_BUDGET)
    return state["used"]


In [ ]:
# 4) 兩階段載入函式（Z-Image-Turbo，bf16）
import torch, gc
from diffusers import ZImagePipeline

MODEL_ID = "Tongyi-MAI/Z-Image-Turbo"
DTYPE = torch.bfloat16
STEPS = 9          # 官方範例 9 步（實際 8 次 DiT forward）；Turbo 模型 guidance 必須為 0

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

def encode_prompts(items):
    """階段 1：只載文字編碼器，把這批 prompt 編成 embeddings 後釋放。回傳 {index: (embeds, n_tokens)}。"""
    pipe = ZImagePipeline.from_pretrained(MODEL_ID, transformer=None, vae=None, torch_dtype=DTYPE, low_cpu_mem_usage=True)
    pipe.to("cuda")
    out = {}
    with torch.no_grad():
        for p in items:
            n_tok = len(pipe.tokenizer(p["prompt"]).input_ids)
            pe, _ = pipe.encode_prompt(prompt=p["prompt"], device="cuda", do_classifier_free_guidance=False, max_sequence_length=512)
            out[int(p["index"])] = ([t.to("cpu") for t in pe], n_tok)
    del pipe; free_gpu()
    return out

def load_generator():
    """階段 2：只載 transformer 與 VAE（不含文字編碼器）。"""
    pipe = ZImagePipeline.from_pretrained(MODEL_ID, text_encoder=None, tokenizer=None, torch_dtype=DTYPE, low_cpu_mem_usage=True)
    pipe.to("cuda")
    pipe.vae.enable_tiling()
    return pipe

def generate_from_embeds(pipe, p, embeds, n_tok):
    claim_image_call()
    w, h = size_for(parse_ratio(p["prompt"]))
    gen = torch.Generator("cpu").manual_seed(int(p["index"]))
    t0 = time.time()
    img = pipe(prompt_embeds=[t.to("cuda") for t in embeds], height=h, width=w,
               num_inference_steps=STEPS, guidance_scale=0.0, generator=gen).images[0]
    return img, {"width": w, "height": h, "seconds": round(time.time() - t0, 1), "tokens": n_tok, "truncated": n_tok > 512}
print("函式已定義")

In [ ]:
# 5) 煙霧測試：編碼 1 個 prompt → 載入生成器 → 出 1 張圖（用完釋放，讓批次重新兩階段載入）
if remaining_image_calls() == 0:
    raise RuntimeError("本輪生圖預算已用完；不載入模型")
t0 = time.time()
p0 = PRESETS[0]
enc = encode_prompts([p0])
print(f"編碼完成 {time.time()-t0:.0f}s，tokens={enc[int(p0['index'])][1]}")
t1 = time.time()
pipe = load_generator()
print(f"生成器載入 {time.time()-t1:.0f}s")
img, info = generate_from_embeds(pipe, p0, *enc[int(p0["index"])])
print(p0["name"], info, f"顯存峰值 {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")
display_img = img.resize((img.width // 2, img.height // 2))
del pipe; free_gpu()
display_img   # 預覽（只縮顯示，不縮存檔）

In [ ]:
# 6) 批次（可續跑，兩階段）：挑出尚未完成者 → 編碼 → 載入生成器 → 逐張出圖並即時存檔
from PIL import Image
manifest = load_manifest()
todo = []
remaining = remaining_image_calls()
if remaining == 0:
    raise RuntimeError("本輪生圖預算已用完；不載入模型")
for p in PRESETS:
    k = manifest_key(p)
    if k in SKIP_KEYS: continue
    if manifest.get(k, {}).get("status") == "ok" and os.path.exists(f"{ORIG}/{filename_for(p)}"): continue
    todo.append(p)
    if len(todo) >= min(MAX_NEW_PER_RUN, remaining): break
print(f"本次要做 {len(todo)} 張（上限 {MAX_NEW_PER_RUN}）")
new_done = errors_in_row = 0
if todo:
    t_start = time.time()
    enc = encode_prompts(todo)
    pipe = load_generator()
    for p in todo:
        k, fn = manifest_key(p), filename_for(p)
        path = f"{ORIG}/{fn}"
        try:
            img, info = generate_from_embeds(pipe, p, *enc[int(p["index"])])
            img.save(path, format="PNG")
            with Image.open(path) as chk:
                assert chk.size == (info["width"], info["height"])
            manifest[k] = {"original": f"preset-originals/colab/{fn}", "width": info["width"], "height": info["height"],
                           "bytes": os.path.getsize(path), "prompt_sha256": sha(p["prompt"]),
                           "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                           "status": "ok", "watermark": False, "model": "Z-Image-Turbo",
                           "truncated": info["truncated"]}
            new_done += 1; errors_in_row = 0
            print(f"[{p['index']:>3}] ok {info['seconds']}s {info['width']}x{info['height']} {k}")
        except Exception as e:
            errors_in_row += 1
            manifest[k] = {"original": None, "prompt_sha256": sha(p["prompt"]), "status": "error", "error": f"{type(e).__name__}: {e}"[:300]}
            print(f"[{p['index']:>3}] ERROR {type(e).__name__}: {str(e)[:120]}")
            free_gpu()
            if errors_in_row >= MAX_CONSEC_ERRORS:
                print("連續失敗，停止；請先查原因再重跑。"); save_manifest(manifest); break
        save_manifest(manifest)
    del pipe; free_gpu()
    print(f"耗時 {(time.time()-t_start)/60:.1f} 分")
ok = sum(1 for v in manifest.values() if v.get("status") == "ok")
print(f"本次新增 {new_done} 張；累計 ok {ok}／{len(PRESETS)}（AGY 已有而跳過 {len(SKIP_KEYS)}）")

In [ ]:
# 7) 打包下載（含 originals 與 manifest）
import shutil
shutil.make_archive("/content/colab_preset_images", "zip", ROOT, ".")
files.download("/content/colab_preset_images.zip")